Schema:

In [0]:
spark.read.parquet(
    "/Volumes/ifood/nyc/yellow_tripdata/"
).printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



Perfil dos dados núméricos que serão utilizandos nas análises:

In [0]:
display(
    spark.read
         .parquet("/Volumes/ifood/nyc/yellow_tripdata/")
         .select(
             "passenger_count",
             "total_amount"
         )
         .describe()
)


---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-4576406347956570>, line 1
----> 1 display(
      2     spark.read
      3          .parquet("/Volumes/ifood/nyc/yellow_tripdata/")
      4          .select(
      5              "passenger_count",
      6              "total_amount"
      7          )
      8          .describe()
      9 )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:100, in Display.display_connect_table(self, df, **kwargs)
     97     self.cf_helper.display_streaming_dataframe(df, config, self.streamin

In [0]:
display(
    spark.read
         .parquet("/Volumes/ifood/nyc/green_tripdata/")
         .select(
             "passenger_count",
             "total_amount"
         )
         .describe()
)

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-4576406347956572>, line 1
----> 1 display(
      2     spark.read
      3          .parquet("/Volumes/ifood/nyc/green_tripdata/")
      4          .select(
      5              "passenger_count",
      6              "total_amount"
      7          )
      8          .describe()
      9 )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:100, in Display.display_connect_table(self, df, **kwargs)
     97     self.cf_helper.display_streaming_dataframe(df, config, self.streaming

In [0]:
display(
    spark.read
         .parquet("/Volumes/ifood/nyc/yellow_tripdata/yellow_tripdata_2023-01.parquet")
         .select(
             "passenger_count",
             "total_amount"
         )
         .describe()
)

summary,passenger_count,total_amount
count,2995023,3066766
mean,1.3625321074328978,27.02038310708492
stddev,0.8961199745510223,22.163588952492074
min,0.0,-751.0
max,9.0,1169.4


In [0]:
display(
    spark.read
         .parquet("/Volumes/ifood/nyc/green_tripdata/green_tripdata_2023-01.parquet")
         .select(
             "passenger_count",
             "total_amount"
         )
         .describe()
)

summary,passenger_count,total_amount
count,63887,68211
mean,1.3158702083365943,21.789377373151172
stddev,0.9790541797737579,15.457114759758213
min,0.0,-71.5
max,9.0,491.0


Perfil dos campos timestamp:

In [0]:
from pyspark.sql.functions import min, max, count, when, col

display(
    spark.read
        .parquet("/Volumes/ifood/nyc/yellow_tripdata/yellow_tripdata_2023-01.parquet")
        .agg(
            min("tpep_pickup_datetime").alias("min_pickup"),
            max("tpep_pickup_datetime").alias("max_pickup"),
            min("tpep_pickup_datetime").alias("min_dropoff"),
            max("tpep_pickup_datetime").alias("max_dropoff"),
            count(when(col("tpep_pickup_datetime").isNull(), 1)).alias("pickup_nuls"),
            count(when(col("tpep_dropoff_datetime").isNull(), 1)).alias("dropoff_nulls")
        )
)

min_pickup,max_pickup,min_dropoff,max_dropoff,pickup_nuls,dropoff_nulls
2008-12-31T23:01:42.000,2023-02-01T00:56:53.000,2008-12-31T23:01:42.000,2023-02-01T00:56:53.000,0,0


In [0]:
from pyspark.sql.functions import min, max, count, when, col

display(
    spark.read
        .parquet("/Volumes/ifood/nyc/green_tripdata/green_tripdata_2023-01.parquet")
        .agg(
            min("lpep_pickup_datetime").alias("min_pickup"),
            max("lpep_pickup_datetime").alias("max_pickup"),
            min("lpep_dropoff_datetime").alias("min_dropoff"),
            max("lpep_dropoff_datetime").alias("max_dropoff"),
            count(when(col("lpep_pickup_datetime").isNull(), 1)).alias("pickup_nuls"),
            count(when(col("lpep_dropoff_datetime").isNull(), 1)).alias("dropoff_nulls")
        )
)

min_pickup,max_pickup,min_dropoff,max_dropoff,pickup_nuls,dropoff_nulls
2009-01-01T20:21:27.000,2023-02-01T03:10:05.000,2009-01-02T11:07:31.000,2023-02-01T17:27:05.000,0,0


Anomalias

In [0]:
%sql
WITH raw AS (
    SELECT 'Yellow' type, date_diff(minute, tpep_pickup_datetime,  tpep_dropoff_datetime) AS trip_minutes, total_amount FROM ifood.bronze.raw_yellow_taxi_tripdata 
    union all 
    SELECT 'Green' type, date_diff(minute,  lpep_pickup_datetime, lpep_dropoff_datetime) as trip_minutes, total_amount FROM ifood.bronze.raw_green_taxi_tripdata
)
select 
    type,
    percentile_approx(total_amount, 0.50) AS total_amount_p50,
     percentile_approx(total_amount, 0.99) AS total_amount_p99,
    percentile_approx(total_amount, 0.999) AS total_amount_p999,
    percentile_approx(trip_minutes, 0.50) AS duration_p50,
    percentile_approx(trip_minutes, 0.99) AS duration_p99,
    percentile_approx(trip_minutes, 0.999) AS duration_p999
from raw where total_amount > 0 and trip_minutes > 0
group by type


type,total_amount_p50,total_amount_p99,total_amount_p999,duration_p50,duration_p99,duration_p999
Yellow,20.7,102.66,167.55,12,65,194
Green,18.5,80.55,142.75,11,63,1408
